# Exploring Stanford PLY meshes with `pyramids-mesh`

The **PLY** format (the Stanford Polygon File Format) was born at the Stanford Computer Graphics Laboratory to store the output of 3D range scanners: a short ASCII or binary header declaring an `element vertex` block and an `element face` block, followed by the vertex coordinates and the polygon vertex-index lists. The classic test objects of computer graphics -- the Stanford *bunny*, *dragon* and *happy buddha* -- all ship this way, as **triangle meshes** reconstructed from overlapping range images in the [Stanford 3D Scanning Repository](http://graphics.stanford.edu/data/3Dscanrep/).

`pyramids-mesh` reads any PLY that carries a **face element** straight into a `pyramids.netcdf.ugrid` object, so a scanner reconstruction lands in the same data model as every other mesh driver. This notebook reads the real Stanford bunny at its coarsest reconstruction, explores its geometry and the per-vertex scanner fields it carries, shows the GIS tools that then come for free, walks the reconstruction-resolution ladder across three models, and ends on the two files the driver deliberately refuses.

## 1. The sample data

The files live one level deeper than the tutorial sample, under `examples/data/real-data/ply/`. Their names are self-describing -- `{model}_{encoding}_{role}_{Nf}f_{Nv}v[_{variant}][_{tags}]` -- so the face/vertex counts and the encoding are readable straight off the filename. We start from the bunny reconstructed at its coarsest resolution (`res4`): small enough to plot instantly, yet a genuine scanner mesh.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from pyramids.netcdf.ugrid import MeshDataset
from cleopatra.basemap.tiles import add_tiles

DATA = Path("../../../examples/data/real-data")
MESH = DATA / "ply" / "bunny_ascii_mesh_948f_453v_res4_vconf-int.ply"
MESH.name, MESH.exists()

`MeshDataset` is the object every driver hands back. Importing from `pyramids` also puts its vendored GDAL/PROJ data in place, so the imports are not order-sensitive here -- we import only the names we use. The `(name, True)` tuple confirms the path resolves.

## 2. Read the bunny in one call

`MeshDataset.read_file` takes the `Path` and an explicit `driver="ply"`. Passing the driver skips suffix sniffing and guarantees the PLY reader is used. The reader also normalises face winding to counter-clockwise as it parses -- on a scanner mesh many triangles come in clockwise, which it reverses (and warns about) so the topology is consistent.

In [ ]:
ds = MeshDataset.read_file(MESH, driver="ply")
ds

The repr shows a `MeshDataset` with 453 nodes and 948 faces, carrying three data variables -- the whole file parsed into topology in a single call. We let cleopatra pick the right matplotlib backend once, immediately after that first read, so every figure below embeds in the cell.

In [ ]:
from cleopatra.config import Config
Config.set_matplotlib_backend()

## 3. Explore the dataset

A handful of attributes describe what we loaded: node and face counts, the coordinate reference system, the names of the data variables, and the spatial extent.

In [ ]:
pd.Series({
    "n_node": ds.n_node,
    "n_face": ds.n_face,
    "crs": ds.crs,
    "epsg": ds.epsg,
    "data_variable_names": ds.data_variable_names,
    "bounds": ds.bounds,
}).to_frame("value")

The bunny has **453 nodes and 948 faces**. `crs`/`epsg` are `None`: PLY stores raw scanner coordinates with no projection metadata, so there is nothing to georeference against (the bounds are in the scanner's own metre-scale units, a few centimetres across). Three data variables came along -- we look at the geometry first, then at those fields.

## 4. The underlying triangulation

`ds.mesh` is the `Mesh2d` object carrying the raw topology: node coordinates and the face-node connectivity table of shape `(n_face, k)`, where row *i* lists the node indices of face *i*. A constant `k` across all rows tells us the polygon order.

In [ ]:
mesh = ds.mesh
fnc = mesh.face_node_connectivity.data
areas = mesh.face_areas
pd.Series({
    "connectivity_shape": fnc.shape,
    "nodes_per_face_k": fnc.shape[1],
    "area_min": float(areas.min()),
    "area_max": float(areas.max()),
}).to_frame("value")

`k = 3` confirms **triangles** -- as expected for a range-scan reconstruction, which triangulates the sampled surface points. The spread between the smallest and largest face area (several orders of magnitude) is the hallmark of a real scan: tiny triangles pack the finely sampled ears while coarser ones span the flanks.

## 5. The mesh outline (wireframe)

`plot_outline()` draws the bare cell edges with no field attached -- the quickest look at the raw geometry the driver produced. These meshes are 3-D surfaces; pyramids works in the plane, so what we see is the triangulation projected onto its *x*/*y* coordinates. We guard the basemap tiles on `ds.epsg`, which is `None` here, so no web tiles are fetched.

In [ ]:
g = ds.plot_outline()
if ds.epsg:
    add_tiles(g.ax, crs=ds.epsg)
g.ax.set_aspect("equal")
g.ax.set_title("Stanford bunny (res4): mesh outline")

The familiar bunny silhouette emerges from the edge soup -- head, ears and haunches recognisable even at 948 triangles. Because we are looking at an *x*/*y* projection of a 3-D surface, the front and back of the model overlap; the dense band of edges is where the curved surface folds toward the viewing plane.

## 6. Per-vertex scanner fields

The bunny reconstructions carry two extra per-vertex properties beyond the vertex coordinates -- `confidence` and `intensity` -- written by the scan-merging pipeline. The driver surfaces them as **node-located data variables** rather than dropping them, alongside `node_z` (the elevation, i.e. the third coordinate). `summary()` returns one row per variable with count/min/max/mean/std.

In [ ]:
ds.summary()

`confidence` ranges roughly 0 to 1 -- the scanner's per-vertex trust in each merged sample, lowest at grazing-angle and seam vertices. `intensity` is the returned laser brightness, and `node_z` is the vertex elevation. All three are defined at the 453 nodes. Because they are node-located, we can colour the mesh by any of them: a true data map, not just geometry.

In [ ]:
gc = ds.plot("confidence", cmap="viridis", colorbar=True,
             title="Stanford bunny: per-vertex scan confidence")
if ds.epsg:
    add_tiles(gc.ax, crs=ds.epsg)
gc.ax.set_aspect("equal")

The map shades each node by its merge confidence. The bright interior is where many scans overlapped and agreed; the darker fringes trace the silhouette and surface folds, where fewer range images saw the surface and the reconstruction is least certain. This is exactly the kind of per-vertex field the driver must preserve -- it is data the file carries, not something pyramids invents.

## 7. GIS operations, for free

Because a `MeshDataset` is a first-class pyramids object, a whole GIS toolkit comes along the moment the file is parsed. First, every face as a row in a GeoDataFrame -- the gateway to the shapely/geopandas ecosystem.

In [ ]:
gdf = ds.to_geodataframe()
print(gdf.shape, list(gdf.columns))
gdf.head()

One row per face (948 of them), each a shapely triangle in the `geometry` column, and no CRS attached -- matching the `crs=None` we saw above. Next, two derived quantities: the dissolved outer footprint, and the per-face circumcentres (defined only for triangular meshes -- which this one is).

In [ ]:
poly = ds.bounding_polygon()
cc = ds.circumcenters()
pd.Series({
    "bounding_polygon_type": type(poly).__name__,
    "bounding_polygon_area": round(poly.area, 6),
    "circumcenters_shape": cc.shape,
}).to_frame("value")

`bounding_polygon()` dissolves the 948 triangles into one shapely polygon -- the mesh footprint -- and `circumcenters()` returns one `(x, y)` per face, succeeding precisely because the mesh is triangular. Both are generic pyramids operations; the PLY driver contributed nothing but the parse.

## 8. The reconstruction-resolution ladder

The repository ships each model at several reconstruction resolutions, which exercise the ASCII parse path at growing sizes. Here we read the coarsest (`res4`) reconstruction of three different models and tabulate their face and vertex counts -- the bunny we already have, plus the dragon and the happy buddha.

In [ ]:
ladder = {
    "bunny (res4)": MESH,
    "dragon (res4)": DATA / "ply" / "dragon_ascii_mesh_11102f_5205v_res4.ply",
    "happy (res4)": DATA / "ply" / "happy_ascii_mesh_15536f_7108v_res4.ply",
}
rows = []
for label, path in ladder.items():
    d = MeshDataset.read_file(path, driver="ply")
    rows.append({"model": label, "n_node": d.n_node, "n_face": d.n_face,
                 "variables": ", ".join(d.data_variable_names)})
pd.DataFrame(rows).set_index("model")

The face count climbs from the bunny's 948 to the dragon's 11,102 and the happy buddha's 15,536 -- the same ASCII reader handling each without a code change. Note the `variables` column: only the bunny reconstructions carry `confidence` + `intensity`; the dragon and happy files store just `node_z`, so the driver surfaces exactly what each file declares, no more.

## 9. What the driver refuses, and why

Not every `.ply` is a mesh. The driver reads a file only when it declares a **face element** whose property is the plain vertex-index list. Two real files in this folder fail that contract, each raising `FileFormatNotSupportedError` with a message explaining the exact reason. We catch both and tabulate the messages rather than letting them abort the notebook.

In [ ]:
from pyramids.base._errors import FileFormatNotSupportedError

rejected = {
    "point cloud (no faces)": "bunny_ascii_cloud_40256v_scan000.ply",
    "binary + per-face scalar": "armadillo_binBE_mesh_345944f_172974v_fscalar-intensity-UNREADABLE.ply",
}
notes = []
for label, name in rejected.items():
    try:
        MeshDataset.read_file(DATA / "ply" / name, driver="ply")
        reason = "(read unexpectedly succeeded)"
    except FileFormatNotSupportedError as err:
        # keep the driver's own explanation, after the 'failed to read ...:' prefix
        reason = str(err).split(": ", 1)[-1]
    notes.append({"file kind": label, "why it is refused": reason})
pd.DataFrame(notes).set_index("file kind")

The first file is a single raw range scan -- 40,256 vertices and **no face element** -- so it is a point cloud, not a mesh, and the driver says so (point clouds belong to `pointclouds`, not here). The second is a binary big-endian armadillo whose face element declares a trailing `intensity` scalar *before* the vertex-index list; the driver refuses it because that extra per-face property would mis-align the face rows. Both are real, common PLY shapes -- the explicit, specific errors are the point: the driver fails loudly with the reason, never silently mis-reads.

## Takeaway

A Stanford scanner reconstruction reads in one line into the same `MeshDataset` as every other format. PLY contributes only the file parsing -- the triangle topology, the `node_z` elevation, and (on the bunny) the per-vertex `confidence` and `intensity` scanner fields, which surface as node-located data variables rather than being dropped. The moment the file is a `MeshDataset`, the full pyramids toolkit applies unchanged: wireframe and field plotting through cleopatra, GeoDataFrame export, bounding polygon, triangular circumcentres. And the driver draws a sharp line around what it accepts -- a file with no face element, or with an unsupported per-face property, is refused with a precise message rather than quietly mis-parsed.